# Question 5 — cached submission

Use the completed search caches to produce `submission_hpc.csv`. This deadline workflow does not search or sweep. Each completed scene is saved and reused after interruption when code, parameters and inputs match.

Use a CPU runtime. Copy the existing `cache_hpc` directory from Torch, or run directly on the allocated Torch compute node. The repository alone does not include those caches.


## 0. Set up the project

The setup cell installs dependencies and finds/clones the project. A fresh clone still needs the completed caches. Do not run this notebook's heavy inference on an HPC login node.


In [ ]:
# Colab setup: install deps and clone this branch if needed.
import os, sys, subprocess
from pathlib import Path

%pip install -q opencv-python-headless numpy scipy pandas pillow

REPO = "https://github.com/rushilara/CVProject1-MerchantsOfSanRamon.git"
BRANCH = "colab-hpc"
IN_COLAB = "google.colab" in sys.modules

if Path("q5_hpc.py").exists():
    PROJECT = Path(".").resolve()
elif IN_COLAB:
    PROJECT = Path("/content/CVProject1-MerchantsOfSanRamon")
    if not (PROJECT / "q5_hpc.py").exists():
        subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(PROJECT)])
else:
    PROJECT = Path(".").resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

needed = [
    "q5_hpc.py", "q5_identify.py", "q5_constellation.py", "q5_search.py", "q5_synth.py",
    "participant/train_ground_truth.csv", "participant/sample_submission.csv",
    "participant/patterns", "participant/train", "participant/validation",
]
missing = [p for p in needed if not Path(p).exists()]
print("cwd", Path.cwd())
if missing:
    raise FileNotFoundError("Upload the project folder. Missing: " + ", ".join(missing))
print("project files ok")

In [ ]:
# Run before importing q5_hpc. Restart the kernel if changing these settings.
import os

os.environ["Q5_FINE"] = "1"
os.environ["Q5_CACHE"] = str(PROJECT / "cache_hpc")  # completed Torch cache directory
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "OPENCV_NUM_THREADS"):
    os.environ[variable] = "1"

import q5_hpc
from q5_identify import load_patterns

print("fine", q5_hpc.FINE, "cache", q5_hpc.CACHE)
patterns = load_patterns(q5_hpc.ROOT / "patterns")
print(len(patterns), "drawings")


## 1. Validate completed caches

Check every validation scene's map shape, patch names and input files. Missing caches stop immediately; no correlation search is started.


In [ ]:
q5_hpc.smoke()


## 2. Use the reviewed defaults

No synthetic parameter sweep and no automatic loading of an old `best_hpc.pkl`. Prediction settings are unchanged from the reviewed defaults. Runtime and leaderboard quality still require measurement; the smoke check only validates inputs.


In [ ]:
print("using", q5_hpc.PARAMS)


## 3. Write the complete submission

Progress is saved under `submission_hpc.checkpoints/`. Rerun this cell to resume. Only after every scene succeeds is `submission_hpc.csv` atomically replaced. Upload the CSV after this cell finishes successfully.


In [ ]:
from pathlib import Path

out = Path("submission_hpc.csv")
q5_hpc.write_submission(patterns, out)
print("wrote", out.resolve())

if "google.colab" in __import__("sys").modules:
    from google.colab import files
    files.download(str(out))